# Intelli-Speech — TEMPORAL_V1 (Phase 3: DSP timing module validation)

This checks that the timing module turns real typical Vaani clips into speech with the timing pattern measured in Experiment 1. That pattern is slower articulation and more, longer and more variable pauses.

The check runs once for each of the 7 leave-one-speaker-out folds:

- **Profiles** are fitted from the other 6 atypical speakers, never from the held-out one.
- **Calibration:** does each synthetic profile match the real speaker it came from?
- **Coverage:** does the held-out real speaker fall inside the range the synthetic speakers span?
- **Pitch:** is it preserved? The output/input F0 ratio should be about 1.

The inputs are read-only: the Exp 1 `features.csv` and `synth_source_v1_full/`. Speakers appear only as S1–S7. Results are written to `analysis_v2/temporal_validation_<date>/`. **Do not commit this notebook with outputs.**

In [ ]:
# CELL 1 | LOGIN + SYNC + DEPENDENCIES
import os, subprocess, sys, json
from pathlib import Path
REPO_DIR = Path("/content/Intelli-Speech")
if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "https://github.com/niharika9nx/Intelli-Speech.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)
os.chdir(REPO_DIR); sys.path.insert(0, str(REPO_DIR))
assert Path("scripts/validate_temporal_transform.py").is_file(), "Pull the latest code first."
!pip install -q soundfile praat-parselmouth "librosa==0.10.2.post1" "numpy<2.5"
DATA_ROOT = Path("/content/drive/MyDrive/Intelli-Speech-data")
FEATURES = sorted((DATA_ROOT / "analysis_v2").glob("exp1_features_*/features.csv"))[-1]
SOURCE = DATA_ROOT / "synth_source_v1_full"
assert (SOURCE / "quality.csv").is_file(), "synth_source_v1_full/quality.csv missing"
print("features:", FEATURES.parent.name, "| source:", SOURCE.name)

In [ ]:
# CELL 2 | RUN VALIDATION (7 folds x 120 clips; ~15-30 min CPU)
!python scripts/validate_temporal_transform.py --features "{FEATURES}" --source-dir "{SOURCE}" --data-root "{DATA_ROOT}"

In [ ]:
# CELL 3 | RESULTS (pseudonymous speakers)
import pandas as pd
OUT = sorted((DATA_ROOT / "analysis_v2").glob("temporal_validation_*"))[-1]
summary = json.loads((OUT / "summary.json").read_text())
for k in ("transforms", "errors", "f0_ratio_quantiles", "insertion_shortfall_total",
          "calibration_outside_control_band_rate", "coverage_rate_overall"):
    print(f"{k:40s} {summary[k]}")
print(); print("calibration |relative error| (median) by feature:")
print(pd.Series(summary["calibration_median_abs_rel_error_by_feature"]).sort_values().to_string())
print(); print("coverage of held-out speaker by feature:")
print(pd.Series(summary["coverage_rate_by_feature"]).sort_values().to_string())
print(); print("coverage by fold:"); print(pd.Series(summary["coverage_rate_by_fold"]).to_string())
cal = pd.read_csv(OUT / "calibration.csv")
print(); print("synthetic vs real (median over folds) for key features:")
key = ["articulation_rate_nuclei_per_s", "pause_rate_per_min_250ms", "pause_p90_s_250ms", "speech_activity_ratio"]
print(cal[cal.feature.isin(key)].groupby(["feature", "profile_source"])[["synthetic", "real_target"]].median().round(3).to_string())

In [ ]:
# CELL 4 | LISTEN TO A FEW EXAMPLES (input vs transformed)
from IPython.display import Audio, display
for wav in sorted((OUT / "examples").glob("*_00_*.wav"))[:6]:
    print(wav.name); display(Audio(str(wav)))

Save the notebook and tell Claude. Do not commit it with outputs.